# About

This jupyter notebook is (one of) the first steps in the pipeline for the human_microbiome_compendium analysis.

Below contains the pipeline for loading the raw data, doing the appropriate filtering, and outputting the dataset files necessary for training & invoking the downstream models.

The dataset name is `v3v4_split_multiproj`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from common.preprocess import *

In [3]:
""" dataset directory. """
DATASET_NAME = "v3v4_split_multiproj_extended"
NOTEBOOK_CACHE = Path("/data/bwh-comppath-seq/youn/human_microbiome_compendium") / DATASET_NAME
NOTEBOOK_CACHE.mkdir(exist_ok=True, parents=True)

# Dataset Files

File locations and metadata.

In [4]:
data_base_dir = Path("/data/cctm/youn/human_microbiome_compendium")

project_metadata_file = data_base_dir / "projects.csv"
asv_sequence_file = data_base_dir / "obs_md.txt.zst"  # tsv format: ASV_NAME    ASV_SEQ, has a header.
abundance_table_dir = data_base_dir / "asv"
sample_metadata_file = data_base_dir / "sample_metadata.tsv"

assert asv_sequence_file.exists(), f"Expected {asv_sequence_file} to exist."
assert asv_sequence_file.is_file(), f"Expected {asv_sequence_file} to be a file."
assert abundance_table_dir.exists(), f"Expected {asv_sequence_file} to exist."
assert abundance_table_dir.is_dir(), f"Expected {abundance_table_dir} to be a directory."
assert sample_metadata_file.exists(), f"Expected {sample_metadata_file} to exist."
assert sample_metadata_file.is_file(), f"Expected {sample_metadata_file} to be a file."

""" Load the project metadata as a pandas dataframe. """
project_metadata = pd.read_csv(project_metadata_file, sep=',')
print("# projects:", project_metadata.shape[0])

""" Load the sample metadata as a pandas dataframe. """
sample_metadata = pd.read_csv(sample_metadata_file, sep='\t')
print("# samples:", sample_metadata.shape[0])

# projects: 482
# samples: 168464


# Projects -- Target Subset by ID

The below project IDs were pre-screened to contain v3-v4 region amplicons, with length at least 400.

In [5]:
PROJECT_IDS = ["PRJNA726866", "PRJNA622517", "PRJNA540406", "PRJNA625181", "PRJNA642894", "PRJEB33905", "PRJEB31155", "PRJNA600229", "PRJNA644097"]
# PROJECT_IDS = ['PRJEB23775', 'PRJNA622517', 'PRJNA428736', 'PRJNA709129', 'PRJNA625181', 'PRJNA630848', 'PRJNA398279', 'PRJNA577051', 'PRJEB31155', 'PRJNA308315', 'PRJEB32537', 'PRJNA401981', 'PRJNA716437', 'PRJDB10612', 'PRJNA540406', 'PRJEB35769', 'PRJNA400325', 'PRJNA414540', 'PRJNA603983', 'PRJNA642894', 'PRJEB42056', 'PRJNA693579', 'PRJNA742936', 'PRJEB33905', 'PRJEB33065', 'PRJNA472768', 'PRJNA388263', 'PRJNA646360', 'PRJEB11419', 'PRJNA547591', 'PRJNA378749', 'PRJNA743361', 'PRJNA544527', 'PRJNA495320', 'PRJDB10527', 'PRJEB6705', 'PRJEB36316', 'PRJNA701870', 'PRJNA726866', 'PRJNA600229', 'PRJNA369083', 'PRJNA528754', 'PRJNA422125', 'PRJNA593062', 'PRJEB6702', 'PRJEB40569']


for project_id in PROJECT_IDS:
    print_project_info(project_id, sample_metadata)

Project: PRJEB23775
	regions: ['unknown']
	isos: ['unknown']
	num samples: 170
Project: PRJNA622517
	regions: ['unknown']
	isos: ['unknown']
	num samples: 78
Project: PRJNA428736
	regions: ['Europe and Northern America']
	isos: ['US']
	num samples: 1201
Project: PRJNA709129
	regions: ['Eastern and South-Eastern Asia']
	isos: ['TH']
	num samples: 168
Project: PRJNA625181
	regions: ['Eastern and South-Eastern Asia']
	isos: ['CN']
	num samples: 166
Project: PRJNA630848
	regions: ['Europe and Northern America']
	isos: ['CA']
	num samples: 50
Project: PRJNA398279
	regions: ['Europe and Northern America']
	isos: ['US']
	num samples: 178
Project: PRJNA577051
	regions: ['Europe and Northern America']
	isos: ['CA']
	num samples: 500
Project: PRJEB31155
	regions: ['unknown']
	isos: ['unknown']
	num samples: 71
Project: PRJNA308315
	regions: ['Europe and Northern America']
	isos: ['US']
	num samples: 80
Project: PRJEB32537
	regions: ['Europe and Northern America']
	isos: ['US']
	num samples: 60
P

# Sample & project filtering.

In [6]:
project_subset = project_metadata.loc[
    project_metadata['project'].isin(PROJECT_IDS), 
    :
]

sample_subset = sample_metadata.loc[
    (
        sample_metadata['project'].isin(PROJECT_IDS)
    )
]
print("[*] Stage 1 filter: {} samples remaining".format(sample_subset.shape[0]))

[*] Stage 1 filter: 17663 samples remaining


In [7]:
project_subset, sample_subset, asv_seqs_subset, sample_max_num_asvs = filter_samples_and_asvs(
    project_subset, sample_subset,
    abundance_table_dir=abundance_table_dir,
    asv_sequence_file=asv_sequence_file,
)

[Project PRJDB10527] Read count statistics:
         read_counts
count      91.000000
mean   143661.780220
std     27889.617773
min     66694.000000
25%    125450.000000
50%    145938.000000
75%    161702.500000
max    203989.000000
[Project PRJDB10527] Using read count threshold of 93219.5 < x < 185246.5
*********************************
[Project PRJDB10612] Read count statistics:
        read_counts
count    117.000000
mean   12145.264957
std     4989.615282
min     5306.000000
25%     9002.000000
50%    10806.000000
75%    13659.000000
max    32130.000000
[Project PRJDB10612] Using read count threshold of 7057.6 < x < 21482.199999999997
*********************************
[Project PRJEB11419] Read count statistics:
         read_counts
count    4850.000000
mean    68171.349072
std    152080.475485
min         1.000000
25%     17341.750000
50%     22169.500000
75%     33531.500000
max    993123.000000
[Project PRJEB11419] Using read count threshold of 7738.8 < x < 534978.0000000001
***

In [8]:
print("# samples:", sample_subset.shape[0])

# samples: 12074


# Process 16S sequences.

Keep only sequences that are actually 16S. (some are 18s by accident!)

In [9]:
ASV_SEQ_PROCESSING_DIR = NOTEBOOK_CACHE / "asv_16s_processing"
ASV_SEQ_PROCESSING_DIR.mkdir(exist_ok=True, parents=True)

"""
Note: defer_to_hpc option makes this pipeline print HPC instructions and raise an error.
Follow the directions, and re-run this cell.
"""
asv_seqs_subset = pipeline_16s_validation(
    asv_seqs_subset,
    cache_dir=ASV_SEQ_PROCESSING_DIR,
    silva_db=data_base_dir / "silva_nr99_v138.2_train_set.fa.gz",
    vsearch_path='vsearch',
    vsearch_num_threads=12,
    min_identity=0.95,
)

Wrote 144419 sequences to /data/bwh-comppath-seq/youn/human_microbiome_compendium/v3v4_split_multiproj_extended/asv_16s_processing/asv_sequences.fasta
VSEARCH found: vsearch v2.30.4_linux_x86_64, 62.5GB RAM, 32 cores
Found 144419 sequences in input file.
Using database: /data/cctm/youn/human_microbiome_compendium/silva_nr99_v138.2_train_set.fa.gz
Minimum identity threshold: 95.0%
Strategy: Keeping only bacterial sequences, filtering out Archaea and Eukaryota

Running VSEARCH on /data/bwh-comppath-seq/youn/human_microbiome_compendium/v3v4_split_multiproj_extended/asv_16s_processing/asv_sequences.fasta against /data/cctm/youn/human_microbiome_compendium/silva_nr99_v138.2_train_set.fa.gz...
VSEARCH output /data/bwh-comppath-seq/youn/human_microbiome_compendium/v3v4_split_multiproj_extended/asv_16s_processing/vsearch_results.tsv already exists!
Parsing VSEARCH results and filtering for bacterial sequences...

RESULTS SUMMARY
Total sequences analyzed: 144419
Confirmed bacterial sequences: 1

In [11]:
""" Run the alignment. """
asv_sequence_file_postblast = ASV_SEQ_PROCESSING_DIR / "asv_sequences.post_filter.fasta"
asv_align_file = ASV_SEQ_PROCESSING_DIR / "asv_alignment.fasta"

dict_to_fasta(asv_seqs_subset, asv_sequence_file_postblast)
run_mafft(asv_sequence_file_postblast, asv_align_file, 'mafft')

NameError: name 'asv_sequence_file_postblast' is not defined

In [ ]:
""" Compute the max sequence length. """
MAX_ASV_SEQUENCE_LEN = max(len(s) for s in asv_seqs_subset.values())
print("Max ASV sequence length:", MAX_ASV_SEQUENCE_LEN)

""" Plot ASV length frequency histogram """
fig, ax = plt.subplots(1, 1)
ax.hist([len(s) for s in asv_seqs_subset.values()], bins=100)
ax.set_xlabel("ASV Sequence length")
ax.set_ylabel("Count")

# Train-test split.

In [ ]:
from common.util import ASVDistanceMatrix

distmat_file = asv_align_file.parent / "distance_matrix.npz"
if distmat_file.exists:
    dist_mat = ASVDistanceMatrix.load(distmat_file)
else:
    dist_mat = ASVDistanceMatrix.from_alignment(asv_align_file)
    dist_mat.save(asv_align_file.parent / "distance_matrix.npz")

In [ ]:
train_df, test_df = train_test_split_mincut_approximation(
    sample_df=sample_subset, 
    abundance_table_dir=abundance_table_dir,
    asv_id_subset=set(asv_seqs_subset.keys()), 
    train_fraction=0.8, 
    test_fraction=0.2,
    distance_matrix=dist_mat,
)
train_df.to_csv(NOTEBOOK_CACHE / "train.tsv", sep="\t", index=False)
test_df.to_csv(NOTEBOOK_CACHE / "test.tsv", sep="\t", index=False)
    
print("# train samples: {}".format(train_df.shape[0]))
print("# test samples: {}".format(test_df.shape[0]))
print("Ratio: {} / {} = {}".format(
    train_df.shape[0], test_df.shape[0], train_df.shape[0] / test_df.shape[0]
))

In [ ]:
train_df.groupby("project")['srs'].count().to_frame()

In [ ]:
test_df.groupby("project")['srs'].count().to_frame()

In [ ]:
project_subset